# STT Lab (Notebook)

Compare speech-to-text models, build a personal voice dataset, LoRA-fine-tune Whisper, and evaluate before/after — without the web UI.

Uses the same Python backend as the FastAPI app (`apps/api`). No HTTP server required.

**Kernel:** choose **STT Lab** (the project venv) if available.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

# Ensure notebooks/helpers.py is importable when launched from repo root or notebooks/
NB_DIR = Path.cwd()
if (NB_DIR / "helpers.py").exists():
    ROOT = NB_DIR.parent
elif (NB_DIR / "notebooks" / "helpers.py").exists():
    ROOT = NB_DIR
    sys.path.insert(0, str(NB_DIR / "notebooks"))
else:
    raise RuntimeError("Run this notebook from stt-lab/ or stt-lab/notebooks/")

import helpers as h

print("Root:", ROOT)
h.models_df()

## 1. Compare models

Set `AUDIO` to a local wav/mp3/webm path and optionally a reference transcript for WER/CER.

In [ ]:
AUDIO = ROOT / "data" / "audio" / "smoke.wav"  # change me
REFERENCE = "hello world"  # or None / ""
MODEL_IDS = ["whisper-tiny", "whisper-base"]  # add cloud ids once keys are set

assert AUDIO.exists(), f"Missing audio file: {AUDIO}"

resp = h.compare(AUDIO, MODEL_IDS, reference=REFERENCE or None)
df = h.results_df(resp)
df

In [ ]:
# Word-level diffs vs reference
for r in resp.results:
    print(f"=== {r.model_name} ===")
    if r.error:
        print("ERROR:", r.error)
        continue
    print("transcript:", r.transcript)
    print("diff:     ", h.show_diff(REFERENCE, r.transcript))
    print()

In [ ]:
# Optional chart: latency vs WER
import matplotlib.pyplot as plt

plot_df = df[df["error"].isna()].copy()
if len(plot_df) and plot_df["wer"].notna().any():
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(plot_df["latency_ms"], plot_df["wer"] * 100)
    for _, row in plot_df.iterrows():
        ax.annotate(row["model"], (row["latency_ms"], row["wer"] * 100), fontsize=8)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("WER (%)")
    ax.set_title("Model contrast")
    plt.show()
else:
    print("Need reference text + successful runs for the WER chart.")

## 2. Build a personal dataset

Save the current clip (or any path) with a ground-truth transcript. Mark at least one `val` sample before fine-tuning.

In [ ]:
DATASET_NAME = "Notebook voice set"
dataset_id = h.create_dataset(DATASET_NAME)
print("dataset_id:", dataset_id)

train_id = h.add_sample(dataset_id, AUDIO, transcript=REFERENCE or "", split="train")
val_id = h.add_sample(dataset_id, AUDIO, transcript=REFERENCE or "", split="val")
print("train sample:", train_id)
print("val sample:  ", val_id)

h.list_datasets()

## 3. Adapt (LoRA fine-tune)

Starts a background training job on Whisper. Prefer small models (`tiny`/`base`) for smoke tests. Expect slow runs on CPU.

In [ ]:
BASE_MODEL = "tiny"
job_id = h.start_finetune(
    dataset_id,
    base_model=BASE_MODEL,
    epochs=1,
    lora_rank=8,
)
print("job_id:", job_id)
status = h.wait_for_job(job_id)
status

## 4. Evaluate before / after

In [ ]:
if status["status"] != "completed":
    raise RuntimeError(f"Fine-tune did not complete: {status}")

ev = h.evaluate(dataset_id, base_model=BASE_MODEL, adapter_id=job_id, split="val")
print(
    f"base WER={ev.base_wer}  adapted WER={ev.adapted_wer}  Δ={ev.delta_wer}  (n={ev.sample_count})"
)
h.eval_df(ev)

## 5. Re-compare including the adapted model

Completed adapters show up as `adapted-<job_id>` in the models table.

In [ ]:
models = h.models_df()
adapted = models[models["provider"] == "adapted"]
print(adapted.to_string(index=False) if len(adapted) else "No adapted models yet")

compare_ids = ["whisper-tiny"]
if len(adapted):
    compare_ids.append(adapted.iloc[0]["id"])

resp2 = h.compare(AUDIO, compare_ids, reference=REFERENCE or None)
h.results_df(resp2)